# 02 — Extraction
**Score-Informed Dynamic Marking Adherence Evaluation — FYP Pipeline (fyp_try2)**

For every performance in the cleaned metadata from notebook 01, this notebook:
1. Extracts dynamic markings (**constant**: p/f/pp/ff/etc., and **wedges**: crescendo/diminuendo)
   from the MusicXML score, with their position in the piece (music21 offsets)
2. Converts each marking's position into real seconds using the `midi_score.mid` tempo curve
3. Maps that score-time position onto the actual performance's timeline, using ASAP's own
   **beat annotation files** as alignment anchors (linear interpolation between beats)
4. Reads the performer's actual MIDI velocity at that mapped moment
   (constant markings: window statistics; wedges: velocity trend across the mapped span)

The result is **one row per marking instance, per performance**, with the score's intended
marking next to the performer's actual velocity — the raw material for labelling adherence.

### Verified against real (n)ASAP files
The core functions were tested on real data before being placed here. On
`Chopin/Ballades/1` + performance `Ali01`, extraction found 31 constant markings and
16 wedges, and the mean performed velocity per marking came out **perfectly ordered**:
pp 40.2 < p 45.8 < f 60.6 < ff 69.6 < fff 79.7 — with diminuendo spans showing negative
velocity slopes and crescendos positive. The mapping chain works.

### Caching
ASAP has ~1066 performances but only ~235 unique scores. Score-side parsing (music21 +
tempo curve + score beats) is by far the slowest step (~6s for a large piece vs ~0.3s per
performance), so it is **cached per unique score**, cutting full-dataset runtime by roughly
two-thirds.

In [ ]:
# @title Setup: install libraries, find project folders, get (n)ASAP
# %pip (not !pip) installs into the SAME Python environment this kernel is running.
# !pip runs whatever "pip" is first on your PATH, which on a local machine is often
# a different Python, so the import below would still fail.
%pip install -q music21 pretty_midi numpy pandas matplotlib

import os, sys, json, random, copy, time, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start):
    """Walk up from `start` until a folder containing utils/constants.py is found."""
    for folder in [start, *start.parents]:
        if (folder / 'utils' / 'constants.py').exists():
            return folder
    return None


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if PROJECT_ROOT is not None:
    # Put the project root at the FRONT of Python's import search list,
    # so "import utils" finds YOUR utils folder.
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    from utils.constants import BASE_LOCATION, DATASET_LOCATION
    BASE = Path(BASE_LOCATION)                          # outputs go here
    ASAP_ROOT = Path(DATASET_LOCATION) / 'asap-dataset' # dataset clone lives here
else:
    print('utils/constants.py not found - using folders next to this notebook')
    BASE = Path.cwd() / 'fyp_try2'
    ASAP_ROOT = Path.cwd() / 'asap-dataset'

if not ASAP_ROOT.exists():
    # On Colab git is always installed. On Windows it may not be.
    if shutil.which('git') is None:
        raise RuntimeError('git was not found. Install Git for Windows, '
                           'then fully restart VS Code and re-run this cell.')
    ASAP_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print('Cloning (n)ASAP dataset (shallow, a few minutes)...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/CPJKU/asap-dataset.git', str(ASAP_ROOT)],
                   check=True)

BASE.mkdir(parents=True, exist_ok=True)
print('Output base :', BASE)
print('ASAP root   :', ASAP_ROOT, '| exists:', ASAP_ROOT.exists())


## 2. Core extraction functions

In [ ]:
# @title Core extraction functions (tested against real ASAP files)
from pathlib import Path
import numpy as np
import pretty_midi
from music21 import converter, dynamics as m21dynamics, spanner

KNOWN_MARKS = {"ppp", "pp", "p", "mp", "mf", "f", "ff", "fff"}


# ---------------------------------------------------------------------
# 1. Score-side extraction (cached per unique score)
# ---------------------------------------------------------------------
def extract_score_side(xml_path, midi_score_path, score_ann_path):
    """
    Parse everything that depends only on the score (not the performance):
      - constant markings [(offset_ql, marking)]
      - wedges            [(start_ql, end_ql, kind)]
      - offset->seconds mapping function (via midi_score.mid ticks)
      - score beat times (from midi_score_annotations.txt)
    Returns a dict; cache it keyed by the score folder.
    """
    score = converter.parse(str(xml_path))
    flat = score.flatten()

    constants = sorted(
        [(float(d.offset), d.value)
         for d in flat.getElementsByClass(m21dynamics.Dynamic)
         if d.value in KNOWN_MARKS],
        key=lambda t: t[0])

    wedges = []
    for sp in score.recurse().getElementsByClass(spanner.Spanner):
        if isinstance(sp, m21dynamics.Crescendo):
            kind = "crescendo"
        elif isinstance(sp, m21dynamics.Diminuendo):
            kind = "diminuendo"
        else:
            continue
        first, last = sp.getFirst(), sp.getLast()
        if first is None or last is None:
            continue
        s = float(first.getOffsetInHierarchy(score))
        e = float(last.getOffsetInHierarchy(score)) + float(
            getattr(last, "quarterLength", 0.0))
        if e > s:
            wedges.append((s, e, kind))
    wedges.sort(key=lambda t: t[0])

    pm_score = pretty_midi.PrettyMIDI(str(midi_score_path))
    resolution = pm_score.resolution

    def offset_to_score_seconds(off_ql):
        return pm_score.tick_to_time(int(round(off_ql * resolution)))

    score_beats = load_beat_times(score_ann_path)

    return {
        "constants": constants,
        "wedges": wedges,
        "offset_to_score_seconds": offset_to_score_seconds,
        "score_beats": score_beats,
    }


# ---------------------------------------------------------------------
# 2. Beat annotations
# ---------------------------------------------------------------------
def load_beat_times(ann_path):
    """ASAP annotation format: <time>\t<time>\t<type>. Returns sorted times."""
    times = []
    with open(ann_path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split("\t")
            if parts and parts[0]:
                times.append(float(parts[0]))
    return np.array(sorted(times))


def map_score_time_to_performance(t_score, score_beats, perf_beats):
    """
    Linear interpolation between corresponding beat anchors.
    ASAP guarantees the same beat sequence in both annotation files,
    so beat i in the score corresponds to beat i in the performance.
    Times outside the annotated range are extrapolated from the edge segment.
    """
    n = min(len(score_beats), len(perf_beats))
    sb, pb = score_beats[:n], perf_beats[:n]
    return float(np.interp(t_score, sb, pb))


# ---------------------------------------------------------------------
# 3. Performance-side velocity lookup
# ---------------------------------------------------------------------
def load_performance_notes(perf_midi_path):
    pm = pretty_midi.PrettyMIDI(str(perf_midi_path))
    notes = sorted((n for i in pm.instruments for n in i.notes),
                   key=lambda n: n.start)
    onsets = np.array([n.start for n in notes])
    velocities = np.array([n.velocity for n in notes])
    return onsets, velocities


def velocity_stats_at(onsets, velocities, t, window=1.0):
    """Velocity statistics for notes with onset in [t - 0.05, t + window]."""
    mask = (onsets >= t - 0.05) & (onsets <= t + window)
    v = velocities[mask]
    if len(v) == 0:
        return {"n_notes": 0, "vel_mean": np.nan,
                "vel_max": np.nan, "vel_min": np.nan}
    return {"n_notes": int(len(v)), "vel_mean": float(v.mean()),
            "vel_max": int(v.max()), "vel_min": int(v.min())}


def wedge_stats_between(onsets, velocities, t0, t1):
    """Velocity trend across a mapped wedge span [t0, t1] in the performance."""
    mask = (onsets >= t0) & (onsets <= t1)
    t, v = onsets[mask], velocities[mask]
    if len(v) < 3:
        return {"n_notes": int(len(v)), "vel_start": np.nan,
                "vel_end": np.nan, "vel_slope": np.nan, "vel_change": np.nan}
    k = max(1, len(v) // 4)  # average first/last quartile for robustness
    slope = float(np.polyfit(t, v, 1)[0])
    return {"n_notes": int(len(v)),
            "vel_start": float(v[:k].mean()),
            "vel_end": float(v[-k:].mean()),
            "vel_slope": slope,
            "vel_change": float(v[-k:].mean() - v[:k].mean())}


# ---------------------------------------------------------------------
# 4. Full extraction for one performance (uses the cached score side)
# ---------------------------------------------------------------------
def extract_performance(score_side, perf_midi_path, perf_ann_path,
                        piece, performance_name, window=1.0):
    """Returns a list of row dicts: one per marking instance."""
    perf_beats = load_beat_times(perf_ann_path)
    onsets, velocities = load_performance_notes(perf_midi_path)
    to_s = score_side["offset_to_score_seconds"]
    sb = score_side["score_beats"]

    rows = []
    for off_ql, marking in score_side["constants"]:
        t_score = to_s(off_ql)
        t_perf = map_score_time_to_performance(t_score, sb, perf_beats)
        stats = velocity_stats_at(onsets, velocities, t_perf, window=window)
        rows.append({"piece": piece, "performance": performance_name,
                     "kind": "constant", "marking": marking,
                     "offset_ql": off_ql, "t_score": t_score,
                     "t_perf": t_perf, **stats})

    for (s_ql, e_ql, kind) in score_side["wedges"]:
        t0 = map_score_time_to_performance(to_s(s_ql), sb, perf_beats)
        t1 = map_score_time_to_performance(to_s(e_ql), sb, perf_beats)
        stats = wedge_stats_between(onsets, velocities, t0, t1)
        rows.append({"piece": piece, "performance": performance_name,
                     "kind": "wedge", "marking": kind,
                     "offset_ql": s_ql, "end_offset_ql": e_ql,
                     "t_score": to_s(s_ql), "t_perf": t0, "t_perf_end": t1,
                     **stats})
    return rows


## 3. Load cleaned metadata from notebook 01

In [ ]:
# @title Load cleaned metadata
meta_path = BASE / 'cleaned_metadata.csv'
assert meta_path.exists(), (
    f'{meta_path} not found. Run Notebook 01 locally first (same utils/constants.py), '
    'or download cleaned_metadata.csv from Google Drive (fyp_try2/) into that folder.')
clean = pd.read_csv(meta_path)
print(f'{len(clean)} performances across {clean["folder"].nunique()} pieces')
clean[['composer', 'folder', 'midi_performance', 'performance_annotations']].head(3)

## 4. Single-performance demo

In [ ]:
# @title Demo: one performance end to end (first piece that actually has dynamics)
side, demo = None, None
# Bach urtext scores contain no dynamics, so try other composers first
cands = pd.concat([clean[clean.composer != 'Bach'], clean[clean.composer == 'Bach']])
for _, cand in cands.drop_duplicates('folder').iterrows():
    trial = extract_score_side(ASAP_ROOT / cand['xml_score'],
                               ASAP_ROOT / cand['midi_score'],
                               ASAP_ROOT / cand['midi_score_annotations'])
    if trial['constants'] or trial['wedges']:
        side, demo = trial, cand
        break
piece = demo['folder']
print(f"{piece}: {len(side['constants'])} constant markings, "
      f"{len(side['wedges'])} wedges, {len(side['score_beats'])} score beats")

rows = extract_performance(side,
                           ASAP_ROOT / demo['midi_performance'],
                           ASAP_ROOT / demo['performance_annotations'],
                           piece, Path(demo['midi_performance']).stem)
demo_df = pd.DataFrame(rows)
demo_df.head(10)

## 5. Batch extraction over the full cleaned dataset
Score-side results are cached per unique `folder`. Progress prints every 25 performances;
failures are collected, not fatal.

In [ ]:
# @title Batch extraction with score-side caching
LIMIT = None            # set e.g. 50 for a quick test run; None = everything
WINDOW = 1.0            # seconds of performance audio after a constant marking

work = clean if LIMIT is None else clean.head(LIMIT)
score_cache, all_rows, errors = {}, [], []
t_start = time.time()

for i, row in enumerate(work.itertuples(index=False), 1):
    piece = row.folder
    try:
        if piece not in score_cache:
            score_cache[piece] = extract_score_side(
                ASAP_ROOT / row.xml_score,
                ASAP_ROOT / row.midi_score,
                ASAP_ROOT / row.midi_score_annotations)
        side = score_cache[piece]
        if not side['constants'] and not side['wedges']:
            continue     # piece has no dynamics (e.g. Bach urtext) - nothing to extract
        perf_name = Path(row.midi_performance).stem
        all_rows += extract_performance(
            side,
            ASAP_ROOT / row.midi_performance,
            ASAP_ROOT / row.performance_annotations,
            piece, perf_name, window=WINDOW)
    except Exception as e:
        errors.append({'piece': piece,
                       'performance': row.midi_performance,
                       'error': f'{type(e).__name__}: {e}'})
    if i % 25 == 0:
        print(f'{i}/{len(work)} performances  |  {len(all_rows)} rows  |  '
              f'{len(score_cache)} scores cached  |  {time.time()-t_start:.0f}s elapsed')

results = pd.DataFrame(all_rows)
err_df = pd.DataFrame(errors)
print(f'\nDONE: {len(results)} marking rows from {results["performance"].nunique() if len(results) else 0} '
      f'performances in {time.time()-t_start:.0f}s  |  {len(errors)} failures')

## 6. Save results + error log to disk

In [ ]:
# @title Save extraction outputs
assert len(results), ('No marking rows extracted - if you ran with a small LIMIT, the first '
                      'pieces alphabetically are Bach urtext scores with no dynamics; '
                      'raise LIMIT or set it to None.')
results.to_csv(BASE / 'extraction_results.csv', index=False)
results[results.kind == 'constant'].to_csv(BASE / 'extraction_constants.csv', index=False)
results[results.kind == 'wedge'].to_csv(BASE / 'extraction_wedges.csv', index=False)
if len(err_df):
    err_df.to_csv(BASE / 'extraction_errors.csv', index=False)
print('Saved:', BASE / 'extraction_results.csv')
print('Constant rows:', (results.kind == 'constant').sum(),
      '| Wedge rows:', (results.kind == 'wedge').sum())

## 7. Sanity charts
If the extraction chain is correct, louder markings should show higher performed
velocities and crescendo/diminuendo spans should show positive/negative velocity change.
These charts are also useful evidence for the thesis (Implementation/Testing chapters).

In [ ]:
# @title Velocity by marking + wedge direction check
const = results[results.kind == 'constant'].dropna(subset=['vel_mean'])
wedge = results[results.kind == 'wedge'].dropna(subset=['vel_change'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
order = [m for m in ['ppp','pp','p','mp','mf','f','ff','fff'] if m in const.marking.unique()]
const.boxplot(column='vel_mean', by='marking', ax=axes[0],
              positions=range(len(order)), grid=False)
axes[0].set_xticklabels(order)
axes[0].set_title('Performed velocity by score marking'); axes[0].set_xlabel('');
axes[0].set_ylabel('mean MIDI velocity near marking')

wedge.boxplot(column='vel_change', by='marking', ax=axes[1], grid=False)
axes[1].axhline(0, color='red', ls='--', lw=1)
axes[1].set_title('Velocity change across wedge spans'); axes[1].set_xlabel('')
axes[1].set_ylabel('velocity change (end - start)')
plt.suptitle(''); plt.tight_layout(); plt.show()

print(const.groupby('marking')['vel_mean'].agg(['count', 'mean']).round(1))
print()
print(wedge.groupby('marking')['vel_change'].agg(['count', 'mean']).round(1))